# Trabajo Final - Curso: Aprendizaje Automático 
Maestría en Ciencia de Datos - UCU  
Equipo: Enrique De Martini, Esteban Cardoso, Lucas Barrios  

In [1]:
import pandas as pd
import numpy as np
import os

## Importar los datasets 
Los dataset son los resultados de la encuesta Aristas 2022 que se realizó a estudiantes de tercer grado de educación media.

#### Dataset de Contexto Socioeconómico

In [ ]:
df_contexto = pd.read_csv(r'https://www.ineed.edu.uy/wp-content/uploads/2023/08/Datos_Estudiante_CXTO.csv',  sep=";",
    encoding="cp1252")

df_contexto = df_contexto.query('IND_CXTO == 1')
df_contexto.describe()


#### Dataset de Habilidades Socioemcionales

In [ ]:
df_socioemocional = pd.read_csv(r'https://www.ineed.edu.uy/wp-content/uploads/2023/08/Datos_Estudiante_Socioemocional.csv',  sep=";",
    encoding="cp1252")

df_socioemocional = df_socioemocional.query('IND_Socioemocional == 1')
df_socioemocional.describe()

### Merge de los dos dataframes

#### Merge según _AlumnoCodigoDes_

In [5]:

# Se eliminan missing values (NaN) en las columnas específicas
df_contexto_limpio = df_contexto.dropna(subset=['Niveles_MAT'])
df_socioemocional_limpio = df_socioemocional.dropna(subset=['Niveles_MAT'])


# Columna indentificadora para hacer el merge
identificador = 'AlumnoCodigoDes'

# Averiguamos qué columnas repetidas hay en los dos dataframes para que no queden ducplicadas en el merge
cols_repetidas = [col for col in df_socioemocional_limpio.columns if col in df_contexto_limpio.columns and col != identificador]

# Eliminamos esas columnas repetidas y hacemos el merge
df_merged = df_contexto_limpio.merge(df_socioemocional_limpio.drop(columns=cols_repetidas), on=identificador, how='inner')

# Se elimina la columna EF10_otro ya que es de llenado manual y puede generar problemas a la hora de la conversion de tipos de datos de object a float
df_merged = df_merged.drop('EF10_otro', axis=1)

#### Conversion de objects a floats

In [6]:
# Se realiza la conversión de las columnas de tipo object a float, reemplazando las comas por puntos y manejando los errores de conversión:

# Seleccionamos automáticamente solo las columnas que son de tipo texto/object
columnas_texto = df_merged.select_dtypes(include=['object']).columns

for col in columnas_texto:
    # Como sabemos que son texto, podemos reemplazar las comas por puntos sin miedo
    df_merged[col] = df_merged[col].str.replace(',', '.')
    
    # Intentamos convertirlas a números. 
    # Usamos errors='ignore' para que si la columna era realmente de texto (ej: nombres de ciudades), 
    # la deje tranquila como texto y no la rompa.
    df_merged[col] = pd.to_numeric(df_merged[col], errors='ignore')

/var/folders/0m/rbf8dr3s1_g2h0t7g9scdbsr0000gn/T/ipykernel_3561/496706323.py:13: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  df_merged[col] = pd.to_numeric(df_merged[col], errors='ignore')


#### Resumen de la auditoria

In [19]:
# Se realiza un resumen de auditoría de los datos.
resumen_auditoria = pd.DataFrame({
    'Tipo_de_Dato': df_merged.dtypes,
    'Valores_Faltantes': df_merged.isnull().sum(),
    # Calculamos la Serie numérica y luego mapeamos el formato de string a cada fila
    'Porcentaje_de_Nulos': ((df_merged.isnull().sum() / len(df_merged)) * 100).map('{:.2f}%'.format),
    'Valores_Unicos': df_merged.nunique(),
    'Moda': df_merged.mode().iloc[0]      
})


# Mostramos la tabla resumen
print(f"Total de filas (Estudiantes que contestaron ambos cuestionarios y que realizaron la prueba de matemática): {df_merged.shape[0]}")
resumen_auditoria

Total de filas (Estudiantes que contestaron ambos cuestionarios y que realizaron la prueba de matemática): 8406


,Tipo_de_Dato,Valores_Faltantes,Porcentaje_de_Nulos,Valores_Unicos,Moda
AlumnoCodigoDes,int64,0,0.00%,8406,1360
GrupoCodigoDes,int64,0,0.00%,611,3434357.0
CentroCodigoDes,int64,0,0.00%,339,60962.0
MdeoInt,object,0,0.00%,2,Interior
regiones,object,0,0.00%,5,SUR
...,...,...,...,...,...
MOTINT_E_ESC50,float64,9,0.11%,591,45.096011
PERSAC_E_ESC50,float64,10,0.12%,1398,50.311376
REGEMO_E_ESC50,float64,18,0.21%,1877,54.561106
VALTIDE_E_ESC50,float64,7,0.08%,541,50.561581


## Eleccion de Variables

#### Diccionario

In [ ]:
# Renombrar las variables que nos interesa 

nombres_legibles = {

    # --- Target ---
    'Niveles_MAT'           : 'Niveles_Matemática',

    # --- Pesos muestrales ---
    'peso_MEst'             : 'w_estudiante',

    # --- Características Comunes (Categóricas) ---
    'MdeoInt'           : 'Montevideo_o_Interior',
    'regiones'          : 'Región',                     # SUR, OESTE, NORTE, ESTE, CENTRO
    'EF46m'             : 'Departamento_residencia',
    'categoria_centro'  : 'Tipo_centro_educativo',      # Secundaria pública, Secundaria privada, UTU (escuela técnica)
    'AlumnoGenero'      : 'Género',
    'EF2d'              : 'Ascendencia', 
    'EF2b'              : 'Pais_de_Nacimiento',
    
    # --- Características Comunes (Numéricas) ---
    'EDAD': 'Edad',
    
    # --- Índices Comunes (Estatus Económico y Social) ---
    'INSE'          : 'Ind_Socioeconómico',
    'ESCS_Alumno'   : 'ESCS_Alumno',
    'ESCS_Centro'   : 'ESCS_Centro',
    'ESCS_Grupo'    : 'ESCS_Grupo',
    
    # --- Índices de Contexto ---
    'ESTSENTPER_E_ESC50'    : 'Ind_Sentido_de_Pertenencia',
    'VINCENTREEST_E_ESC50'  : 'Ind_Vínculo_entre_Estudiantes',
    'VINESTADS_E_ESC50'     : 'Ind_Vínculo_con_Adscriptos',
    'VINESTPROF_E_ESC50'    : 'Ind_Vínculo_con_Profesores',
    'VOZESTU_E_ESC50'       : 'Ind_Voz_Estudiantil',
    'ACTITUDLEC_E_ESC50'    : 'Ind_Actitud_hacia_la_Lectura',
    'ACTITUDMAT_E_ESC50'    : 'Ind_Actitud_hacia_la_Matemática',
    
    # --- Índices de Habilidades Socioemocionales ---
    'AUTOCON_E_ESC50'       : 'Ind_de_Autocontrol',
    'AUTOEIDE_E_ESC50'      : 'Ind_de_Autoeficacia_académica_en_Idioma_Español',
    'AUTOEMAT_E_ESC50'      : 'Ind_de_Autoeficacia_académica_en_Matemática',
    'AUTOMETA_E_ESC50'      : 'Ind_de_Autorregulación_metacognitiva',
    'EMPATIA_E_ESC50'       : 'Ind_de_Empatía',
    'EXTERNALIZ_E_ESC50'    : 'Ind_de_Conductas_externalizantes',
    'HABINTER_E_ESC50'      : 'Ind_de_Habilidades_interpersonales',
    'HABINTRA_E_ESC50'      : 'Ind_de_Habilidades_intrapersonales',
    'HABRELAC_E_ESC50'      : 'Ind_de_Habilidades_de_relacionamiento',
    'INTERNALIZ_E_ESC50'    : 'Ind_de_Conductas_internalizantes',
    'MOTAUTREGA_E_ESC50'    : 'Ind_de_Motivación_y_autorregulación_del_aprendizaje',
    'MOTINT_E_ESC50'        : 'Ind_de_Motivación_intrínseca',
    'PERSAC_E_ESC50'        : 'Ind_de_Perseverancia_académica',
    'REGEMO_E_ESC50'        : 'Ind_de_Regulación_emocional',
    'VALTIDE_E_ESC50'       : 'Ind_de_Valoración_de_la_tarea_en_Idioma_Español',
    'VALTMAT_E_ESC50'       : 'Ind_de_Valoración_de_la_tarea_en_Matemática'
}

#### Filtrado de las variables y binarizacion del Target 

In [12]:
# Filtrar y renombrar en un solo paso
df_procesado = df_merged[list(nombres_legibles.keys())].rename(columns=nombres_legibles)

# Nuevo mapeo de target para volverla binaria (1 = B1, N1, N2; 0 = N3, N4, N5)
mapeo_target = {
    'B1': 1,
    'N1': 1,
    'N2': 1,
    'N3': 0,
    'N4': 0,
    'N5': 0
}

# Aplicamos el mapeo 
df_procesado['Mal_desempeño'] = df_procesado['Niveles_Matemática'].map(mapeo_target)

# Eliminamos la columna original
df_procesado = df_procesado.drop(columns=['Niveles_Matemática'])

## Split en Train y Test de los datos 

In [ ]:
from sklearn.model_selection import train_test_split
X = df_procesado.drop(columns=['Mal_desempeño', 'w_estudiante'])   # Características
y = df_procesado['Mal_desempeño']                                  # Target
w = df_procesado['w_estudiante']                                   # Pesos muestrales


X_train, X_test, y_train, y_test, w_train, w_test = train_test_split(
    X, 
    y, 
    w,
    test_size=0.2, 
    random_state=42,
    stratify=y 
)